EXPERIMENT - 5 AND 6

Pune House Price Prediction Using Machine Learning

The real estate market in Pune has experienced rapid growth due to IT hubs, educational institutions, and expanding infrastructure. Property prices vary significantly depending on location, amenities, connectivity, and environmental factors.

A real estate analytics company aims to build a that can accurately predict the median house price in different localities of Pune.

You are provided with a dataset containing information about various housing features collected from multiple areas such as Hinjewadi, Kothrud, Baner, Wakad, and Viman Nagar.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:
df = pd.read_csv("housing.csv")
df.head()

df.info()
df.describe()

df.isnull().sum()

In [ ]:
plt.figure(figsize=(10,8))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm')
plt.show()

sns.histplot(df['MEDV'], kde=True)
plt.title("House Price Distribution")
plt.show()

df.fillna(df.mean(), inplace=True)

In [ ]:
Q1 = df.quantile(0.25)
Q3 = df.quantile(0.75)
IQR = Q3 - Q1

df = df[~((df < (Q1 - 1.5 * IQR)) | (df > (Q3 + 1.5 * IQR))).any(axis=1)]

In [ ]:
X = df.drop('MEDV', axis=1)  # features
y = df['MEDV']               # target

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
lr = LinearRegression()
lr.fit(X_train, y_train)

dt = DecisionTreeRegressor(random_state=42)
dt.fit(X_train, y_train)

rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

gb = GradientBoostingRegressor(random_state=42)
gb.fit(X_train, y_train)

In [ ]:
def evaluate(model, X_test, y_test):
    y_pred = model.predict(X_test)

    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    return mae, mse, r2

In [ ]:
models = {
    "Linear Regression": lr,
    "Decision Tree": dt,
    "Random Forest": rf,
    "Gradient Boosting": gb
}

results = {}

for name, model in models.items():
    results[name] = evaluate(model, X_test, y_test)

results_df = pd.DataFrame(results, index=['MAE', 'MSE', 'R2']).T
print(results_df)

In [ ]:
importances = rf.feature_importances_
feature_names = X.columns

feat_imp = pd.Series(importances, index=feature_names)
feat_imp.sort_values().plot(kind='barh', figsize=(8,6))
plt.title("Feature Importance")
plt.show()